# Structured Responses and Prompting Techniques

LLMs return **free-form text** by default. That's a problem when you need to feed their output into code — you need **reliable, parseable structure**.

This notebook covers three levels of making LLM output structured, plus a key prompting technique:

| Technique | Reliability | How |
|---|---|---|
| Ask nicely for JSON | Fragile | Prompt only |
| JSON mode | Good | API parameter |
| Pydantic validation | Solid | Parse + validate in code |

And a prompting technique:

| Technique | Purpose |
|---|---|
| Chain-of-thought | Get the model to reason step by step before answering |

In [ ]:
import json
from openai import OpenAI

client = OpenAI(
    base_url='http://localhost:11434/v1',
    api_key='ollama',
)

MODEL = "gemma4:e2b"

TEXT = """Dr. Marie Dupont is a professor at HEVS in Sion, Switzerland.
She has been teaching Natural Language Processing since 2018 and has published
over 40 papers. Her research focuses on multilingual transformers and
low-resource languages. She previously worked at Google Research in Zurich
from 2012 to 2017."""

## 1. Naive: "Please return JSON"

We ask the model to return JSON in the prompt. This often works... but the model may wrap it in markdown code fences, add commentary, or produce invalid JSON.

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    temperature=0,
    messages=[{
        "role": "user",
        "content": f"""
Extract structured information from this text. 
Return JSON with keys: name, title, institution, location, research_topics (list), publication_count (int). 
Important: Strip any preamble like ```json etc. Output just the JSON array

Text: {TEXT}"""
    }],
)

raw = response.choices[0].message.content
print(raw)
print("\n" + "=" * 60)

# Try to parse it as JSON — this will likely fail
try:
    data = json.loads(raw)
    print("Parsed OK:", data)
except json.JSONDecodeError as e:
    print(f"JSON parse FAILED: {e}")
    print("The model probably wrapped JSON in ```json ... ``` or added extra text.")

## 2. JSON Mode

The OpenAI API (and Ollama) support `response_format={"type": "json_object"}`, which forces the model to output **valid JSON only** — no markdown fences, no commentary.

In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    temperature=0,
    response_format={"type": "json_object"},
    messages=[{
        "role": "user",
        "content": f"""Extract structured information from this text. Return JSON with keys:
name, title, institution, location, research_topics (list), publication_count (int).

Text: {TEXT}"""
    }],
)

raw = response.choices[0].message.content
print(raw)
print("\n" + "=" * 60)

data = json.loads(raw)
print("Parsed OK!")
print(json.dumps(data, indent=2))

JSON mode guarantees **valid JSON**, but it does NOT guarantee the **schema** is correct. The model might use different key names, nest things differently, or return `publication_count` as a string instead of an int.

## 3. Pydantic Validation

[Pydantic](https://docs.pydantic.dev/) lets us define the **exact schema** we expect and validate the model's output against it.

In [ ]:
from pydantic import BaseModel

class ResearcherProfile(BaseModel):
    name: str
    title: str
    institution: str
    location: str
    research_topics: list[str]
    publication_count: int


In [ ]:
response = client.chat.completions.create(
    model=MODEL,
    temperature=0,
    response_format={
        "type": "json_schema",
        "json_schema": {
            "schema": ResearcherProfile.model_json_schema(),
            "strict": True,
        }
    },
    messages=[{"role": "user", "content": f"Extract: {TEXT}"}],
)

profile = ResearcherProfile.model_validate_json(response.choices[0].message.content)

print(f"Name:         {profile.name}")
print(f"Title:        {profile.title}")
print(f"Institution:  {profile.institution}")
print(f"Location:     {profile.location}")
print(f"Topics:       {profile.research_topics}")
print(f"Publications: {profile.publication_count} (type: {type(profile.publication_count).__name__})")